# DS2002 · Public API to Clean Table

**Lab — 2026-10-09 · Fall 2026**  

---

## Lab 07 — Public API to Clean Table

Build the weather-ingestion function you will actually reuse on the midterm. It has to pull real data, survive a flaky network, say plainly when it fell back to sample data, and hand back a table that is safe to join.

The last requirement is the one people skip. A function that returns *something* no matter what is worse than one that fails loudly, because the failure ends up in your findings instead of your console.

### Part 1 — a fetch you can trust

**TODO:** finish `get_weather`. Requirements:

- sets a timeout
- retries on 429 and 5xx with an increasing delay
- raises immediately on 400 or 404 — those are your bug, not theirs
- returns a DataFrame from the `daily` block

You wrote this shape in Wednesday's studio. Rebuild it from memory if you can.

In [1]:
import requests, pandas as pd, time

BASE = 'https://archive-api.open-meteo.com/v1/archive'
RETRY_STATUSES = {429, 500, 502, 503, 504}

def get_weather(start, end, lat=38.03, lon=-78.51, tries=3):
    """Daily max temperature and total precipitation for a date range."""
    params = {
        'latitude': lat, 'longitude': lon,
        'start_date': start, 'end_date': end,
        'daily': 'temperature_2m_max,precipitation_sum',
        'timezone': 'America/New_York',
    }
    # TODO: loop up to `tries` times
    # TODO:   request with a timeout; return pd.DataFrame(r.json()['daily']) on 200
    # TODO:   raise on a status that is not in RETRY_STATUSES
    # TODO:   otherwise sleep (increasing each time) and try again
    # TODO: raise if every attempt failed

    for attempt in range(tries):
        try:
            r = requests.get(BASE, params=params, timeout=10)
            if r.status_code == 200:
                return pd.DataFrame(r.json()['daily'])
            if r.status_code not in RETRY_STATUSES:
                r.raise_for_status()
            r.raise_for_status()
        except requests.exceptions.RequestException:
            if r.status_code not in RETRY_STATUSES or attempt == tries - 1:
                raise
            time.sleep(2 ** attempt)
    pass

### Part 2 — two home weekends

Pull weather for **2026-09-05** and **2026-10-03**. One range covering both is fine and is one request instead of two — be nice to free APIs.

If the call genuinely fails, use the fallback in Part 3, but you must record that you did.

In [2]:
from numpy import exceptions
# TODO: call get_weather once for a range that spans both dates,
#       then keep just the two dates you care about
used_fallback = False
try:
    weather = get_weather('2026-09-05', '2026-10-03')
    weather = weather[weather['time'].isin(['2026-09-05', '2026-10-03'])]
    wx = weather.copy()
    wx['source'] = 'open-meteo'
except requests.exceptions.RequestException:
    used_fallback = True
    print("API request failed. Fallback data needed in Part 3.")
wx

,time,temperature_2m_max,precipitation_sum,source
0,2026-09-05,29.4,2.5,open-meteo
28,2026-10-03,22.0,14.7,open-meteo


### Part 3 — the fallback, labelled as a fallback

Sample values so the notebook still runs without a network. The important part is the `source` column: whoever reads your results can tell which rows are real.

In [3]:
FALLBACK = pd.DataFrame({
    'time': ['2026-09-05', '2026-10-03'],
    'temperature_2m_max': [30.1, 18.4],
    'precipitation_sum': [0.0, 22.5],
})

# TODO: if you used the fallback, set used_fallback = True and build `wx` from FALLBACK
# TODO: add a 'source' column to `wx` -- 'open-meteo' or 'fallback sample'

### Part 4 — make it joinable

**TODO:** produce a frame with a real datetime column named `date`, plus `temp_max`, `precip_mm`, and `source`. Rename the API's column names to yours — your pipeline should not inherit somebody else's naming.

In [4]:
# TODO
wx_clean = wx.rename(columns={
    'time': 'date',
    'temperature_2m_max': 'temp_max',
    'precipitation_sum': 'precip_mm',
})
wx_clean['date'] = pd.to_datetime(wx_clean['date'])
wx_clean = wx_clean[['date', 'temp_max', 'precip_mm', 'source']]
wx_clean

,date,temp_max,precip_mm,source
0,2026-09-05,29.4,2.5,open-meteo
28,2026-10-03,22.0,14.7,open-meteo


### Part 5 — validate before you use it

This is the habit that separates a pipeline from a script. State what must be true, and find out here rather than three cells later inside a join.

**TODO:** uncomment and make them pass.

In [5]:
wx = wx_clean.copy()

assert list(wx.columns) == ['date', 'temp_max', 'precip_mm', 'source']
assert len(wx) == 2, f'expected 2 rows, got {len(wx)}'
assert pd.api.types.is_datetime64_any_dtype(wx['date'])
assert wx['precip_mm'].notna().all(), 'missing precipitation'
assert (wx['precip_mm'] >= 0).all(), 'negative rainfall is not a thing'
print('weather table is safe to join.')

weather table is safe to join.


### Part 6 — answer the question

**TODO:** which of the two dates was wetter, and by how much? Print a sentence a vendor could read, not just a number.

In [6]:
# TODO
wetter = wx.loc[wx['precip_mm'].idxmax()]
drier = wx.loc[wx['precip_mm'].idxmin()]
difference = wetter['precip_mm'] - drier['precip_mm']
print(f"{wetter['date']:%B %d, %Y} was wetter than {drier['date']:%B %d, %Y} with a difference of {difference:.1f} mm precipitation.")


October 03, 2026 was wetter than September 05, 2026 with a difference of 12.2 mm precipitation.


### Part 7 — cite it

**TODO:** print the endpoint, the coordinates, the date range, when you pulled it, and whether the data is real or fallback.

In [7]:
# TODO
from datetime import datetime
print(f"API endpoint: {BASE}")
print("Coordinates: 38.03, -78.51")
print(f"Date range: 2026-09-05 to 2026-10-03")
print(f"Date retrieved: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Date source: {'fallback sample' if used_fallback else 'open_meteo (real API data)'}")

API endpoint: https://archive-api.open-meteo.com/v1/archive
Coordinates: 38.03, -78.51
Date range: 2026-09-05 to 2026-10-03
Date retrieved: 2026-10-12 01:09:58
Date source: open_meteo (real API data)


### Write-up

Suppose `get_weather` returned an empty DataFrame on failure instead of raising. Walk through what happens next: you merge it onto 200 days of sales and chart rain against revenue. What does the chart show, and why would nobody catch it?

_If get_weather returned an empty DataFrame instead of raising an error, we might not realize that the weather data was never collected. When we merge it with the 200 days of sales data, the sales records would still be there if we used a left join, but the rainfall values would be missing. If we replaced those missing values with zeros, the chart would make it seem like there was no rain during those days. This could make us think that rainfall has no effect on revenue, even though we never had the actual weather data to compare. Nobody might notice because the code would still run and produce a chart without showing an error. This is why we need to check that the data is complete and accurate before using it to make conclusions._

### Stretch

Pull the same two dates for a second location — Orlando is at roughly `28.54, -81.38` — and put both cities in one long-format frame with a `city` column. That is the exact shape the midterm needs when you compare stores in different places.

In [8]:
# stretch
orlando = get_weather('2026-09-05', '2026-10-03', lat=28.54, lon=-81.38)
orlando = orlando[orlando['time'].isin(['2026-09-05', '2026-10-03'])]
orlando = orlando.rename(columns={
    'time': 'date',
    'temperature_2m_max': 'temp_max',
    'precipitation_sum': 'precip_mm'
})
orlando['date'] = pd.to_datetime(orlando['date'])
orlando['source'] = 'open-meteo'
orlando['city'] = 'Orlando'
charlottesville = wx.copy()
charlottesville['city'] = 'Charlottesville'
combined = pd.concat([charlottesville, orlando], ignore_index=True)
combined = combined[['date', 'city', 'temp_max', 'precip_mm', 'source']]
combined

,date,city,temp_max,precip_mm,source
0,2026-09-05,Charlottesville,29.4,2.5,open-meteo
1,2026-10-03,Charlottesville,22.0,14.7,open-meteo
2,2026-09-05,Orlando,32.1,3.9,open-meteo
3,2026-10-03,Orlando,30.4,14.5,open-meteo
